# Lab 1 · PINN fundamentals

양끝이 고정된 줄의 변위를 구합니다. 이어서 줄의 길이를 바꿔 보고, 몇 곳에서 측정한 변위만으로 하중을 얼마나 정확히 추정할 수 있는지 살펴봅니다.

[Start Here](../../00_Start_Here.ipynb) · Previous: [Introduction](../../01_Introduction.ipynb) · Next: [Lab 2: Projectile motion](../02_projectile/Lab_2_Projectile_Motion.ipynb)

The three runs use the same equation, $u''(x)=f(x)$.

| Run | What you know | What the network predicts |
|---|---|---|
| Forward | Constant load and zero displacement at each end | Displacement $u(x)$ |
| Parameterized | Load, length, and endpoint conditions | Displacement $u(x,L)$ for a given length $L$ |
| Inverse | Displacement at 100 points and the endpoint conditions | Displacement $u(x)$ and the unknown load $f(x)$ |

### Domain and training loop

![Lab 1 domains: fixed interval, parameterized lengths, and inverse observations](../../ETC/assets/teaching/lab1-domain.svg)

The forward and parameterized runs use fixed grids. The dots in the inverse problem represent displacement measurements.

![Lab 1 code flow: model predictions and a separate PDE object feed PhysicsInformer and the L-BFGS loss](../../ETC/assets/teaching/lab1-flow.svg)

Follow `BasicPINN`, `Poisson1D`, `loss_terms`, and `optimize_lab` in [pinn_basics.py](source_code/pinn_basics.py). The shared [mlp / informer helpers](../../ETC/runtime/labs.py) construct the PhysicsNeMo components. The inverse networks use the local `FourierMLP` class.

<a id="first-problem"></a>
## Lab 1.1 · Forward problem

On $0\leq x\leq1$, the displacement satisfies $u''(x)=1$ and $u(0)=u(1)=0$. A fully connected network takes a position $x$ and predicts $u_\theta(x)$. Its smooth activation function lets us differentiate the prediction twice with respect to $x$.

The first term in the loss measures how closely the prediction follows the equation at 256 evenly spaced points, including the endpoints. The second term measures the displacement at the two ends.

$$L=\frac{1}{N}\sum_{i=1}^{N}\left(u_\theta''(x_i)-1\right)^2 + 10\left[u_\theta(0)^2+u_\theta(1)^2\right].$$

Integrating the equation twice and applying the endpoint conditions gives $u(x)=x(x-1)/2$. The training code uses this curve to evaluate the result. You can compare it with your own prediction after training.

![Analytical solution and PINN displacement between the fixed endpoints](images/single_parabola.png)

### Defining the equation

SymPy's `Symbol`, `Function` and `.diff` describe the equation. `PDE`, imported from `physicsnemo.sym.eq.pde`, stores its named residual; the training loop evaluates the residual.

```python
class Poisson1D(PDE):
    def __init__(self, inverse=False):
        self.dim = 1
        x = Symbol("x")
        u = Function("u")(x)
        f = Function("f")(x) if inverse else 1
        self.equations = {"poisson": u.diff(x, 2) - f}
```

### Follow one training step

Open [pinn_basics.py](source_code/pinn_basics.py) beside [the shared Lab helpers](../../ETC/runtime/labs.py). `BasicPINN` calls the local `mlp` helper, which constructs `physicsnemo.models.mlp.FullyConnected` with tanh activations. A batch `(N, 1)` of positions gives `(N, 1)` solution values. In parameterized mode the network receives two features, normalized position and normalized length, and still returns one value per point. `setup` supplies configuration, seed and device; `save_run` writes artifacts.

The local `informer(Poisson1D(...), device)` constructs `physicsnemo.sym.eq.phy_informer.PhysicsInformer` with `grad_method="autodiff"`. Trace the dictionary `{'coordinates': x, 'u': u}` into `physics.forward`: its output `['poisson']` is the residual at each point. `loss_terms` squares and reduces that tensor and adds the boundary penalty.

| Operation | Find it in the Python files | Meaning in this problem |
|---|---|---|
| Coordinate to prediction | `BasicPINN.forward` in [pinn_basics.py](source_code/pinn_basics.py) | A batch of $x$ values produces $u_\theta(x)$. |
| Spatial derivatives and residual | `loss_terms` calls `PhysicsInformer` | Differentiate with respect to $x$ to evaluate $u_{\theta,xx}-1$. |
| PDE and boundary losses | `loss_terms` returns `physics` and `boundary` | Penalize interior residuals and $u_\theta(0)^2+u_\theta(1)^2$; forward and parameterized endpoints use soft penalties with weight 10. The inverse example below enforces the endpoints exactly. |
| Parameter gradients and update | `optimize_lab` in [pinn_basics.py](source_code/pinn_basics.py) | PyTorch's `torch.optim.LBFGS(model.parameters(), ...)` uses `optimizer.step(closure)` to recompute the loss and `loss.backward()` on the same fixed training points, then updates the weights and biases. |

Spatial derivatives define the equation residual. Parameter gradients determine the weight update. The solution should be convex and nonpositive between its zero endpoints. Why does the equation alone allow an extra $ax+b$? Which loss term removes that freedom?

<a id="forward-pinn-execution"></a>
### Train the forward model

Run the next three cells in order. The first sets the device and number of training steps, the second trains the model, and the third plots the result. The default is 1,000 L-BFGS optimizer calls in FP32. One call can evaluate the loss several times during its line search.

Compare the PINN curve with the analytical solution and check the errors on the finer evaluation grid in `metrics.json`. To try a different network, edit [pinn_basics.py](source_code/pinn_basics.py) and rerun the training and plot cells.

In [ ]:
import os
import sys
import subprocess
import uuid
from pathlib import Path
import numpy as np
from IPython import get_ipython
get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "00_Start_Here.ipynb").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the workshop repository.")
sys.path.insert(0, str(ROOT))
from ETC.runtime.notebook import show_results, validate_settings, completed_output
LAB = ROOT / "01_labs/01_pinn"
OUTPUT_BASE = Path(os.environ.get("AI4SCI_OUTPUT_DIR", str(LAB / "outputs"))).expanduser().resolve()
DEVICE = os.environ.get("AI4SCI_DEVICE", "auto")
STEPS = int(os.environ.get("AI4SCI_STEPS", "1000"))  # Full-batch L-BFGS optimizer calls
RUN_DIRS = {}
RUN_COMPLETED = {}
validate_settings(DEVICE, STEPS)
print("PhysicsNeMo target: 2.2.2", "device:", DEVICE, "steps:", STEPS)

In [ ]:
RUN_COMPLETED["forward"] = False
OUTPUT = OUTPUT_BASE / ("forward-" + uuid.uuid4().hex[:8])
RUN_DIRS["forward"] = OUTPUT
command = [sys.executable, str(LAB / "source_code/pinn_basics.py"), "--device", DEVICE,
           "--steps", str(STEPS), "--seed", "42", "--output-dir", str(OUTPUT)] + ["--mode", "forward"]
validate_settings(DEVICE, STEPS)
subprocess.run(command, check=True, cwd=ROOT)
RUN_COMPLETED["forward"] = True
metrics = show_results(OUTPUT, steps=STEPS, seed=42, preview=False)

In [ ]:
OUTPUT = completed_output(RUN_DIRS, RUN_COMPLETED, "forward")
data = np.load(OUTPUT / "predictions.npz", allow_pickle=False)
plt.plot(data["x"], data["reference"], label="analytical")
plt.plot(data["x"], data["prediction"], label="PINN")
plt.xlabel("x"); plt.legend(); plt.show()

## Lab 1.2: Parameterized problems

How does the solution change when the right endpoint moves? Add the domain length $l\in[1,2]$ as a network input and solve
$$
\begin{align}
    \mathbf{P} : \left\{\begin{matrix}
\frac{\mathrm{d}^2 u}{\mathrm{d} x^2}(x) = f(x), \\
\\
u(0) = u(l) = 0,
\end{matrix}\right.
\end{align}
$$
The network now takes both position and length as inputs, $u_{net}(x,l)$.
The L-BFGS run uses a fixed grid: 17 equally spaced lengths in $[1,2]$, with 32 midpoint positions in $[0,l]$ for each length by default. Weight the residual by $l_i$ to account for each interval's length when approximating the domain integral.

$$L_{residual}\approx\frac1N\sum_{i=1}^N l_i\left(u_{xx}(x_i,l_i)-1\right)^2,$$
$$L_{BC}\approx\frac1N\sum_{i=1}^N\left[u(0,l_i)^2+u(l_i,l_i)^2\right].$$

The analytical solution is $u(x,l)=x(x-l)/2$. Fixed evaluation uses $l=1,1.25,1.5,1.75,2$ and reports pooled errors plus `heldout_before/after.per_length`. These five lengths are also in the training grid, so the checks do not establish accuracy at new lengths. Use `per_length` to compare lengths. The preview and `validation_rmse` are for $l=1.5$.

![Reference illustration of parameterized solutions](images/every_parabola.png)

In [ ]:
RUN_COMPLETED["parameterized"] = False
OUTPUT = OUTPUT_BASE / ("parameterized-" + uuid.uuid4().hex[:8])
RUN_DIRS["parameterized"] = OUTPUT
command = [sys.executable, str(LAB / "source_code/pinn_basics.py"), "--device", DEVICE,
           "--steps", str(STEPS), "--seed", "42", "--output-dir", str(OUTPUT)] + ["--mode", "parameterized"]
validate_settings(DEVICE, STEPS)
subprocess.run(command, check=True, cwd=ROOT)
RUN_COMPLETED["parameterized"] = True
metrics = show_results(OUTPUT, steps=STEPS, seed=42, preview=False)

In [ ]:
OUTPUT = completed_output(RUN_DIRS, RUN_COMPLETED, "parameterized")
data = np.load(OUTPUT / "predictions.npz", allow_pickle=False)
plt.plot(data["x"], data["reference"], label="analytical")
plt.plot(data["x"], data["prediction"], label="PINN")
plt.xlabel("x"); plt.legend(); plt.show()

## Lab 1.3 · Inverse problem

Suppose the load $f(x)$ is unknown. You have 100 measurements of the displacement on $0<x<1$ and know that both endpoints are fixed. A solution network estimates $u(x)$ from the measurements. A second network estimates $f(x)$, and the equation $u''(x)=f(x)$ couples their predictions.

For this experiment the measurements come from the analytical displacement

$$u(x)=\frac{x^3-x}{6}-\frac{\sin(4\pi x)}{16\pi^2}.$$

The corresponding load is $f(x)=x+\sin(4\pi x)$. The first plot compares the estimated and reference loads. The second shows the displacement and the observations. Compare these examples with the plots from your own run below.

![Reference and learned source](images/inverse_parabola.png)

![Learned displacement and the observations used to fit it](images/inverse_parabola_2.png)

### Inverse PINN execution

Train both networks together using the same 100 fixed, noise-free observations of `u` throughout the run. Only displacement is observed.

Write `u(x) = x(1-x) * v(x)` so both zero boundary values hold exactly. Fit the observations in the equivalent form:

```python
v_observed = u_observed / (x_observed * (1 - x_observed))
data_loss = 1000 * mean((v_predicted - v_observed)**2)
```

Only interior observations are used in this division. It weights near-boundary observations more strongly, where small solution errors can hide large curvature errors. The total loss is the PDE residual MSE plus `data_loss`. This scaling is for the supplied noise-free example; it can amplify measurement noise near the boundaries.

Both networks receive smooth coordinate features `[2*x-1, sin(k*pi*x), cos(k*pi*x)]` for `k=1,2,3,4`, followed by two tanh layers of width 32. The solution factor is `v = 0.1 * N_u`; the source is `f = N_f`. The feature set includes frequencies relevant to this example, but no source coefficients or source values are supplied.

The inverse model uses `FourierMLP`, built from `torch.nn.Linear`, `Tanh` and `Sequential`. Each network maps the same `(N, 1)` positions to one field. `loss_terms` supplies both `u` and `f` to the same PhysicsInformer, now evaluating `u_xx - f`; L-BFGS updates both networks through `model.parameters()`.

Run 1,000 L-BFGS optimizer calls, then inspect **both** curves. The accuracy check includes `source_rmse` and `source_max_abs` over 401 points, including the endpoints. Compare the source error with the displacement error near each endpoint.

In [ ]:
RUN_COMPLETED["inverse"] = False
OUTPUT = OUTPUT_BASE / ("inverse-" + uuid.uuid4().hex[:8])
RUN_DIRS["inverse"] = OUTPUT
command = [sys.executable, str(LAB / "source_code/pinn_basics.py"), "--device", DEVICE,
           "--steps", str(STEPS), "--seed", "42", "--output-dir", str(OUTPUT)] + ["--mode", "inverse"]
validate_settings(DEVICE, STEPS)
subprocess.run(command, check=True, cwd=ROOT)
RUN_COMPLETED["inverse"] = True
metrics = show_results(OUTPUT, steps=STEPS, seed=42, preview=False)

In [ ]:
OUTPUT = completed_output(RUN_DIRS, RUN_COMPLETED, "inverse")
data = np.load(OUTPUT / "predictions.npz", allow_pickle=False)
plt.plot(data["x"], data["reference"], label="analytical")
plt.plot(data["x"], data["prediction"], label="PINN")
plt.xlabel("x"); plt.legend(); plt.show()

In [ ]:
plt.plot(data["x"], data["source_reference"], label="true f(x)")
plt.plot(data["x"], data["source_prediction"], label="inferred f(x)")
plt.legend(); plt.show()

### Try a different setting

- Reduce the number of L-BFGS calls and compare the displacement and source errors with the 1,000-call run.
- In the inverse problem, vary the number of observations or `INVERSE_DATA_WEIGHT` in [pinn_basics.py](source_code/pinn_basics.py). Where does the largest source error occur?
- In the parameterized problem, evaluate a length between two training lengths and compare the prediction with the analytical curve.

`--config` loads the training settings from YAML. An explicit `--steps` argument overrides the value in the file.

[Start Here](../../00_Start_Here.ipynb) · Previous: [Introduction](../../01_Introduction.ipynb) · Next: [Lab 2: Projectile motion](../02_projectile/Lab_2_Projectile_Motion.ipynb)

--- 

Further reading: [Open Hackathons Resources](https://www.openhackathons.org/s/technical-resources). Community support: [OpenACC and Hackathons Slack Channel](https://www.openacc.org/community#slack).

---

# Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials may include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.